# Optimisation des Coûts de Transport
## Étape 2 : Analyse et Simulation de Scénarios

Après l'exploration, place à l'optimisation. Dans ce notebook, je calcule les métriques d'efficacité (coût par kg), j'identifie les aberrations statistiques (expéditions sur-coûteuses), et je réalise une simulation de basculement de fret pour estimer nos économies potentielles.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

data_path = "../data/SCMS_Delivery_History_Dataset.csv"
df = pd.read_csv(data_path)

# Nettoyage basique
df['Weight (Kilograms)'] = pd.to_numeric(df['Weight (Kilograms)'].replace('Weight Captured Separately', np.nan), errors='coerce')
df['Freight Cost (USD)'] = pd.to_numeric(df['Freight Cost (USD)'].replace('Freight Included in Commodity Cost', np.nan).replace('Invoiced Separately', np.nan), errors='coerce')
df = df.dropna(subset=['Shipment Mode', 'Weight (Kilograms)', 'Freight Cost (USD)'])

df['Delivered to Client Date'] = pd.to_datetime(df['Delivered to Client Date'], errors='coerce')
df['PO Sent to Vendor Date'] = pd.to_datetime(df['PO Sent to Vendor Date'], errors='coerce')
df['Delay_Days'] = (df['Delivered to Client Date'] - df['PO Sent to Vendor Date']).dt.days


### 1. Calcul de l'efficacité : Coût Moyen par Kg
C'est la métrique de base de la Supply Chain pour comparer des pommes avec des pommes.

In [ ]:
df['Cost_per_Kg'] = df['Freight Cost (USD)'] / df['Weight (Kilograms)']

efficiency = df.groupby('Shipment Mode')['Cost_per_Kg'].mean().sort_values(ascending=False)
print("Coût Moyen par Kilogramme (USD) :\n", efficiency)

### 2. Détection des aberrations (Expéditions sur-coûteuses)
Je cherche les expéditions dont le coût est astronomique par rapport à la moyenne du même mode de transport (moyenne + 2 écarts-types).

In [ ]:
def detect_outliers(group):
    mean_cost = group['Freight Cost (USD)'].mean()
    std_cost = group['Freight Cost (USD)'].std()
    threshold = mean_cost + (2 * std_cost)
    return group[group['Freight Cost (USD)'] > threshold]

outliers = df.groupby('Shipment Mode').apply(detect_outliers).reset_index(drop=True)
print(f"Nombre d'expéditions anormalement chères détectées : {len(outliers)}")
print(f"Surcoût total généré par ces anomalies : ${outliers['Freight Cost (USD)'].sum():,.2f}")

### 3. Le Simulateur : Réduction des Coûts
Et si nous imposions une règle stricte : Remplacer 20% des expéditions aériennes par du maritime (Ocean). Quelle serait l'économie ?

In [ ]:
air_df = df[df['Shipment Mode'] == 'Air']
ocean_avg_cost_per_kg = df[df['Shipment Mode'] == 'Ocean']['Cost_per_Kg'].mean()
ocean_avg_delay = df[df['Shipment Mode'] == 'Ocean']['Delay_Days'].mean()

# On prend 20% des expéditions aériennes au hasard
sim_air_to_ocean = air_df.sample(frac=0.20, random_state=42)

original_cost = sim_air_to_ocean['Freight Cost (USD)'].sum()

# Le nouveau coût est calculé avec le taux au kilo du maritime
simulated_ocean_cost = (sim_air_to_ocean['Weight (Kilograms)'] * ocean_avg_cost_per_kg).sum()

savings = original_cost - simulated_ocean_cost

print("--- RÉSULTATS DE LA SIMULATION (Air -> Ocean 20%) ---")
print(f"Coût original aérien : ${original_cost:,.2f}")
print(f"Nouveau coût maritime estimé : ${simulated_ocean_cost:,.2f}")
print(f"\nÉconomie brute réalisée : ${savings:,.2f}")
print(f"Nouveau délai moyen estimé pour ces colis : {ocean_avg_delay:.0f} jours")

## Recommandations Business

1. **Chasse au Gaspillage** : L'algorithme a isolé des expéditions dont le prix dépasse de 2 écarts-types la moyenne. Il y a eu des surfacturations ou des envois d'urgence (type "Air Charter"). Bloquer ces anomalies en amont avec une validation managériale permettrait une économie instantanée.
2. **Stratégie de Basculement (Shift)** : Mon simulateur prouve que migrer seulement 20% de notre fret aérien vers l'océan rapporte une économie massive. 
3. **Trade-off Coût / Délai** : Ce basculement va mécaniquement augmenter les délais de livraison. L'action recommandée est d'anticiper les commandes (meilleur forecasting) pour pouvoir se permettre le délai maritime sans tomber en rupture de stock.